# Hyperparameter Sweep — Optuna (Latent JacobianODE)

This notebook launches a **Bayesian hyperparameter sweep** for the Latent JacobianODE model
using [Optuna](https://optuna.org/) via the `hydra-optuna-sweeper` plugin.

**How it differs from the grid sweep notebook:**
- Instead of exhaustive grid search (`itertools.product`), Optuna uses **TPE (Tree-structured Parzen Estimators)** to sample promising hyperparameter regions.
- Fewer trials needed to find good configurations (sample-efficient).
- Search spaces are continuous ranges (log-uniform, uniform) rather than discrete lists.
- Optional SQLite storage for **resumable** sweeps across SLURM failures.

**Workflow:**
1. Configure fixed params + Optuna search space (Sections 1–4)
2. Build and launch the `--multirun sweeper=optuna` command (Section 5)
3. Analyze results via Optuna study + W&B (Section 6)
4. Apply post-hoc physics selection (Section 7)

## 0. Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

import wandb

# Ensure we're in the repo root
REPO_ROOT = Path(os.environ.get("JACOBIANODE_ROOT", ".." )).resolve()
os.chdir(REPO_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /orcd/home/002/eisenaj/code/JacobianODE


In [3]:
# ----------------------------------------------------------------
# Paths and W&B settings
# ----------------------------------------------------------------
SAVE_DIR     = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs"
WANDB_ENTITY = "JacobianODE"

## 1. Training Mode & Encoder Settings

Set `MODE` to `"from_scratch"` or `"pretrained"`.

In [4]:
# ============================================================
# Training mode
# ============================================================
MODE = "from_scratch"  # <-- CHANGE THIS

assert MODE in ("from_scratch", "pretrained"), f"Invalid MODE: {MODE}"

# Encoder architecture (from_scratch only)
ENCODER_TYPE = "spline_coupling"  # "mlp" | "coupling" | "spline_coupling"

print(f"Training mode: {MODE}, Encoder: {ENCODER_TYPE}")

Training mode: from_scratch, Encoder: spline_coupling


## 2. Data Settings

In [5]:
# ============================================================
# Data source
# ============================================================
DATA_SOURCE = "dysts"  # "dysts" | "wmtask"

# ---- dysts settings ----
FLOW_TARGET   = "JacobianODE.dysts_sim.flows.Lorenz"
N_PERIODS     = 12
PTS_PER_PERIOD = 100
NUM_ICS       = 32
OBS_NOISE     = 0.01
NORMALIZE     = True
# OBSERVED_INDICES = "all"  # list of ints or "all"
OBSERVED_INDICES = [0]

# ---- Delay embedding ----
N_DELAYS      = 100
DELAY_SPACING = 1

# ---- wmtask settings (only if DATA_SOURCE == "wmtask") ----
WMTASK_PROJECT = "WMSelectionTask__cue_time_0.1__response_time_0.25__enforce_fixation_False"
WMTASK_NAME = "BiologicalRNN__cue_time_0.1__learning_rate_0.0005__max_epochs_42__N1_64__N2_64__tau_0.05__dt_0.02__eig_lower_bound_0.1__init_mode_random"
WMTASK_MODEL_TO_LOAD = "final"
WMTASK_DATALOADER = "all"
WMTASK_TRAJ_WINDOW = "delay2"   # 'delay2' or 'full'
WMTASK_DIM = 128                # N1 + N2 = 64 + 64

## 3. Encoder-Specific Settings (from_scratch)

In [6]:
# ============================================================
# From-scratch encoder architecture
# ============================================================
if MODE == "from_scratch":
    # Latent dimension
    RECONSTRUCTION_MODE      = "most_recent" # 'uniform' | 'harmonic' | 'most_recent'

    # Encoder warmup
    ENCODER_WARMUP_EPOCHS  = 5
    DYNAMICS_WARMUP_EPOCHS = 0

    # Learnable loss weights
    LEARN_R2_WEIGHT           = False
    LEARN_LOOP_CLOSURE_WEIGHT = False
    LEARN_FNN_WEIGHT          = False
    LEARN_JAC_CONS_WEIGHT     = False
    LEARN_JAC_NORM_WEIGHT     = False
    LOG_VAR_INIT              = 0.0

    DECODE_ONLY_RECENT = False

    if ENCODER_TYPE == "mlp":
        N_LATENT = 3
        MLP_HIDDEN_DIM = 128
        MLP_N_LAYERS   = 4
        MLP_DROPOUT    = 0.0

        # Decoder
        DECODER_HIDDEN     = 128
        DECODER_LAYERS     = 3

    elif ENCODER_TYPE in ("coupling", "spline_coupling"):
        N_LATENT = None  # computed from data dim in the config-building cell
        N_TARGET_DIMS       = 3  

        # ---- Architecture ----
        N_COUPLING_LAYERS   = 8
        COUPLING_HIDDEN_DIM = 128
        N_HIDDEN_LAYERS     = 2
        ZERO_INIT           = True   # identity initialization trick
        PERMUTATION_SEED    = 42     # base seed for fixed random permutations
        USE_LOFT            = False  # LOFT layer after coupling blocks
        LOFT_TAU            = 100.0  # LOFT threshold

        # ---- VAE settings ----
        USE_VAE                = True    # VAE reparameterization on dynamic subspace
        VAE_SAMPLE_ALL_LOSSES  = False    # when False, only recon uses sampled z_dyn
        KL_WARMUP_EPOCHS       = 0       # 0 = fixed weight, >0 = linear ramp
        KL_NULL_WEIGHT      = "null"
        KL_DYN_WEIGHT       = 0.0001  # default if not swept

        # ---- Tangent entropy loss ----
        TANGENT_ENTROPY_WEIGHT   = 0.0
        TANGENT_ENTROPY_MODE     = "quadratic"
        TANGENT_ENTROPY_N_SAMPLES = 1024

        if ENCODER_TYPE == "coupling":
            # Coupling-specific
            SCALE_ACTIVATION    = "tanh" # bounds log-scale via tanh(·)
            SCALE_CLAMP         = 3.0    # max |log_s|, ~20× scaling range
            # ---- Numerical stability (Andrade 2024, arXiv:2402.16408) ----
            CLAMP_TYPE          = "symmetric"  # 'symmetric' | 'asymmetric'
            ALPHA_POS           = 0.1          # asymmetric clamp: expansion bound
            ALPHA_NEG           = 2.0          # asymmetric clamp: compression bound

        # Spline-coupling-specific
        if ENCODER_TYPE == "spline_coupling":
            NUM_BINS    = 8
            TAIL_BOUND  = 3.0
            USE_ACTNORM = True

## 4. Optuna Search Space & Sweep Settings

Define which hyperparameters Optuna should optimize and their ranges.

**Search space types:**
- `float` with `log: True` — log-uniform sampling (good for loss weights, learning rates)
- `float` with `log: False` — uniform sampling
- `int` — uniform integer sampling
- `categorical` with `choices: [...]` — discrete choices

In [7]:
# ============================================================
# Optuna sweep configuration
# ============================================================

# Number of trials and parallelism
N_TRIALS         = 40    # total Optuna trials
N_PARALLEL_JOBS  = 10     # concurrent GPU worker jobs
N_STARTUP_TRIALS = 10    # random trials before TPE kicks in
SAMPLER_SEED     = 42

# ---- Early pruning ----
# At OPTUNA_PRUNE_EPOCH, each worker reads the SQLite study and compares
# its trajectory val_loss against completed trials.  If worse than the
# configured quantile, training stops early (saving ~90% of that trial's
# GPU time).  Set to None to disable.
OPTUNA_PRUNE_EPOCH         = 15    # epoch to evaluate (None = no pruning)
OPTUNA_PRUNE_MIN_COMPLETED = 5     # need this many finished trials first
OPTUNA_PRUNE_QUANTILE      = 0.5   # prune if worse than median

# ---- Constrained optimization ----
# When enabled, trials that violate the constraint are considered infeasible.
# A feasible result always beats an infeasible one (regardless of loss).
# Set to None to disable (use unconstrained progress tracking).
OPTUNA_CONSTRAINT_METRIC    = "val/loop_closure_loss"  # metric to check feasibility
OPTUNA_CONSTRAINT_THRESHOLD = N_TARGET_DIMS ** 0.5     # sqrt(n_dynamic_dims)

# ---- Search space ----
# Keys must match Hydra config paths.
# Each entry is passed to hydra.sweeper.search_space.
SEARCH_SPACE = {
    "training.lightning.loop_closure_weight": {
        "type": "float", "low": 1e-7, "high": 1.0, "log": True,
    },
    "training.lightning.kl_dyn_weight": {
        "type": "float", "low": 1e-7, "high": 1.0, "log": True,
    },
    # ---- Uncomment to add more params to the search ----
    "training.lightning.optimizer_kwargs.lr": {
        "type": "float", "low": 1e-6, "high": 1e-3, "log": True,
    },
    # "training.lightning.fnn_weight": {
    #     "type": "float", "low": 0.0, "high": 1.0, "log": False,
    # },
    # "training.lightning.tangent_entropy_weight": {
    #     "type": "float", "low": 1e-4, "high": 1e-1, "log": True,
    # },
}

print(f"Optuna sweep: {N_TRIALS} trials, {N_PARALLEL_JOBS} parallel jobs")
print(f"Pruning: epoch={OPTUNA_PRUNE_EPOCH}, quantile={OPTUNA_PRUNE_QUANTILE}, min_completed={OPTUNA_PRUNE_MIN_COMPLETED}")
if OPTUNA_CONSTRAINT_METRIC:
    print(f"Constraint: {OPTUNA_CONSTRAINT_METRIC} <= {OPTUNA_CONSTRAINT_THRESHOLD:.4f}")
else:
    print("Constraint: disabled (unconstrained)")
print(f"Search space ({len(SEARCH_SPACE)} params):")
for k, v in SEARCH_SPACE.items():
    print(f"  {k}: {v}")

Optuna sweep: 40 trials, 10 parallel jobs
Pruning: epoch=15, quantile=0.5, min_completed=5
Constraint: val/loop_closure_loss <= 1.7321
Search space (3 params):
  training.lightning.loop_closure_weight: {'type': 'float', 'low': 1e-07, 'high': 1.0, 'log': True}
  training.lightning.kl_dyn_weight: {'type': 'float', 'low': 1e-07, 'high': 1.0, 'log': True}
  training.lightning.optimizer_kwargs.lr: {'type': 'float', 'low': 1e-06, 'high': 0.001, 'log': True}


## 5. Fixed Parameters & Overrides

These parameters are held constant across all Optuna trials.

In [8]:
# ============================================================
# Fixed training params (not swept by Optuna)
# ============================================================
PREDICTION_STEPS = 30

FIXED_PARAMS = {
    "model.prediction_steps":                                    PREDICTION_STEPS,
    "training.batch_size":                                       32 if MODE == "from_scratch" else 16,
    "training.trainer_params.max_epochs":                        150 if MODE == "from_scratch" else 200,
    "training.trainer_params.limit_train_batches":               200,
    "training.trainer_params.limit_val_batches":                 50,
    "training.trainer_params.accumulate_grad_batches":           4,
    "training.lightning.optimizer_kwargs.lr":                    1e-4,
    "training.lightning.optimizer_kwargs.weight_decay":          1e-4,
    "training.lightning.scheduler_type":                         "cosine",
    "training.lightning.reconstruction_loss_weight":             1.0,
    "training.lightning.latent_prediction_loss_weight":          1.0,
    "training.lightning.jac_consistency_weight":                 0.0,
    "training.lightning.fnn_normalize":                          True,
    "training.lightning.fnn_elementwise_regularization":         True,
    "training.lightning.fnn_use_pca":                            True,
    "training.lightning.fnn_n_samples":                          1024,
    "training.lightning.jacobianODEint_kwargs.traj_init_steps":  15,
    "training.lightning.jacobianODEint_kwargs.interp_pts":       4,
    "training.lightning.jacobianODEint_kwargs.inner_N":          20,
    "training.lightning.jacobianODEint_kwargs.inner_path":       "line",
    "training.early_stopping.early_stopping_patience":           5,
    "training.early_stopping.min_epochs":                        10,
    "training.early_stopping.percent_thresh":                    0.01,
    "training.model_checkpoint.save_top_k":                      1,
}

# Remove swept params from FIXED_PARAMS if they overlap
for key in SEARCH_SPACE:
    FIXED_PARAMS.pop(key, None)

print(f"{len(FIXED_PARAMS)} fixed params, {len(SEARCH_SPACE)} swept params")

23 fixed params, 3 swept params


In [9]:
# ============================================================
# W&B settings
# ============================================================
WANDB_ENTITY = "JacobianODE"

# Auto-generate project name (matches grid sweep convention)
_obs_idx_label = ''.join(str(i) for i in OBSERVED_INDICES) if OBSERVED_INDICES != "all" else "all"
_data_prefix = "Lorenz" if DATA_SOURCE == "dysts" else "WMTask"
_warmup = ENCODER_WARMUP_EPOCHS if MODE == "from_scratch" else "pt"

if MODE == "from_scratch":
    if ENCODER_TYPE in ("coupling", "spline_coupling"):
        _latent_label = f"T{N_TARGET_DIMS}"
        _enc_label = "spline_coupling" if ENCODER_TYPE == "spline_coupling" else "coupling"
    else:
        _latent_label = f"L{N_LATENT}"
        _enc_label = "MLP"
    WANDB_PROJECT = f"{_data_prefix}_IND{_obs_idx_label}_N{N_DELAYS}_D{DELAY_SPACING}_Norm{NORMALIZE}_{_latent_label}__{_enc_label}__JacobianODE"
else:
    WANDB_PROJECT = f"{_data_prefix}_Pretrained_L{N_LATENT}__JacobianODE"

WANDB_PROJECT_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}"

# W&B group name with datetime tag
from datetime import datetime
_datetime_tag = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
_swept_keys_short = "_".join(k.split(".")[-1] for k in SEARCH_SPACE)
WANDB_GROUP = f"optuna_{MODE}_{_enc_label}_{_swept_keys_short}_n{N_TRIALS}_{_datetime_tag}"

# Auto-generated study name (matches W&B group)
STUDY_NAME = WANDB_GROUP

OPTUNA_SAVE_DIR = Path(SAVE_DIR) / "optuna"

# Study persistence — SQLite DB so the sweep survives coordinator restarts
# and completed trials are never lost. Set to None only for quick local tests.
OPTUNA_STORAGE = f"sqlite:///{OPTUNA_SAVE_DIR}/optuna_{STUDY_NAME}.db"

print(f"W&B project:    {WANDB_PROJECT_PATH}")
print(f"W&B group:      {WANDB_GROUP}")
print(f"Optuna study:   {STUDY_NAME}")
print(f"Optuna storage: {OPTUNA_STORAGE}")

W&B project:    JacobianODE/Lorenz_IND0_N100_D1_NormTrue_T3__spline_coupling__JacobianODE
W&B group:      optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50
Optuna study:   optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50
Optuna storage: sqlite:////orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/optuna/optuna_optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50.db


## 5. Build & Preview Command

In [10]:
# ============================================================
# Build overrides
# ============================================================

def _fmt(v):
    """Format a value for Hydra CLI override."""
    if isinstance(v, bool):
        return str(v).lower()
    if v is None:
        return "null"
    return str(v)


# Fixed param overrides
overrides = [f"{k}={_fmt(v)}" for k, v in FIXED_PARAMS.items()]

# Jacobian MLP settings
JAC_HIDDEN_DIMS = [128, 128]
JAC_NUM_LAYERS  = 2
JAC_ACTIVATION  = "silu"
JAC_WINDOW_STRIDE = 5 if MODE == "from_scratch" else PREDICTION_STEPS
# No quotes around list — coordinator uses single-run mode, not --multirun
_hidden_dim_str = "[" + ",".join(str(d) for d in JAC_HIDDEN_DIMS) + "]"
overrides.append(f"model.params.hidden_dim={_hidden_dim_str}")
overrides.append(f"model.params.num_layers={JAC_NUM_LAYERS}")
overrides.append(f"model.params.activation={JAC_ACTIVATION}")

# Sequence length (matches grid sweep)
TRAJ_INIT_STEPS = FIXED_PARAMS.get(
    "training.lightning.jacobianODEint_kwargs.traj_init_steps", 15
)
JAC_WINDOW = TRAJ_INIT_STEPS + PREDICTION_STEPS
SEQ_LENGTH = JAC_WINDOW

# ---- Data overrides ----
if DATA_SOURCE == "dysts":
    # No quotes around list — coordinator uses single-run mode, not --multirun
    _obs_idx_str = "[" + ",".join(str(i) for i in OBSERVED_INDICES) + "]" if OBSERVED_INDICES != "all" else "all"
    overrides += [
        "data=dysts",
        f"data.flow._target_={FLOW_TARGET}",
        f"data.trajectory_params.n_periods={N_PERIODS}",
        f"data.trajectory_params.pts_per_period={PTS_PER_PERIOD}",
        f"data.trajectory_params.num_ics={NUM_ICS}",
        f"data.postprocessing.obs_noise={OBS_NOISE}",
        f"data.postprocessing.normalize={NORMALIZE}",
        f"data.train_test_params.delay_embedding_params.observed_indices={_obs_idx_str}",
        f"data.train_test_params.delay_embedding_params.n_delays={N_DELAYS}",
        f"data.train_test_params.delay_embedding_params.delay_spacing={DELAY_SPACING}",
        f"data.train_test_params.seq_length={SEQ_LENGTH}",
    ]
elif DATA_SOURCE == "wmtask":
    overrides += [
        "data=wmtask",
        f"data.dataset_loader.project={WMTASK_PROJECT}",
        f"data.dataset_loader.name={WMTASK_NAME}",
        f"data.dataset_loader.model_to_load={WMTASK_MODEL_TO_LOAD}",
        f"data.dataset_loader.dataloader_to_use={WMTASK_DATALOADER}",
        f"data.dataset_loader.traj_window={WMTASK_TRAJ_WINDOW}",
        f"data.flow.dim={WMTASK_DIM}",
    ]

# ---- Encoder overrides (from_scratch) ----
if MODE == "from_scratch":
    # Compute n_input (n_delays × n_raw_obs)
    _n_raw_obs = len(OBSERVED_INDICES) if OBSERVED_INDICES != "all" else (3 if DATA_SOURCE == "dysts" else WMTASK_DIM)
    _n_input = N_DELAYS * _n_raw_obs

    if ENCODER_TYPE == "mlp":
        overrides += [
            "model=latent_mlp",
            f"model.encoder.n_input={_n_input}",
            f"model.encoder.n_latent={N_LATENT}",
            f"model.encoder.hidden_dim={MLP_HIDDEN_DIM}",
            f"model.encoder.n_layers={MLP_N_LAYERS}",
            f"model.encoder.dropout={MLP_DROPOUT}",
            f"model.encoder.decoder_hidden={DECODER_HIDDEN}",
            f"model.encoder.decoder_layers={DECODER_LAYERS}",
            f"model.encoder.decoder_n_output={_n_raw_obs if DECODE_ONLY_RECENT else 'null'}",
            "model.encoder.context_margin=0",
        ]
    elif ENCODER_TYPE == "coupling":
        overrides += [
            "model=latent_coupling",
            f"model.encoder.n_input={_n_input}",
            f"model.encoder.n_coupling_layers={N_COUPLING_LAYERS}",
            f"model.encoder.hidden_dim={COUPLING_HIDDEN_DIM}",
            f"model.encoder.n_hidden_layers={N_HIDDEN_LAYERS}",
            f"model.encoder.scale_activation={SCALE_ACTIVATION}",
            f"model.encoder.scale_clamp={SCALE_CLAMP}",
            f"model.encoder.zero_init={_fmt(ZERO_INIT)}",
            f"model.encoder.permutation_seed={PERMUTATION_SEED}",
            f"model.encoder.clamp_type={CLAMP_TYPE}",
            f"model.encoder.alpha_pos={ALPHA_POS}",
            f"model.encoder.alpha_neg={ALPHA_NEG}",
            f"model.encoder.use_loft={_fmt(USE_LOFT)}",
            f"model.encoder.loft_tau={LOFT_TAU}",
            f"model.n_target_dims={_fmt(N_TARGET_DIMS)}",
            f"model.use_vae={_fmt(USE_VAE)}",
            f"model.vae_sample_all_losses={_fmt(VAE_SAMPLE_ALL_LOSSES)}",
            f"model.kl_warmup_epochs={KL_WARMUP_EPOCHS}",
            f"training.lightning.kl_null_weight={KL_NULL_WEIGHT}",
            f"training.lightning.reconstruction_mode={RECONSTRUCTION_MODE}",
            f"training.lightning.tangent_entropy_weight={TANGENT_ENTROPY_WEIGHT}",
            f"training.lightning.tangent_entropy_mode={TANGENT_ENTROPY_MODE}",
            f"training.lightning.tangent_entropy_n_samples={TANGENT_ENTROPY_N_SAMPLES}",
        ]
    elif ENCODER_TYPE == "spline_coupling":
        overrides += [
            "model=latent_spline_coupling",
            f"model.encoder.n_input={_n_input}",
            f"model.encoder.n_coupling_layers={N_COUPLING_LAYERS}",
            f"model.encoder.hidden_dim={COUPLING_HIDDEN_DIM}",
            f"model.encoder.n_hidden_layers={N_HIDDEN_LAYERS}",
            f"model.encoder.num_bins={NUM_BINS}",
            f"model.encoder.tail_bound={TAIL_BOUND}",
            f"model.encoder.use_actnorm={_fmt(USE_ACTNORM)}",
            f"model.encoder.zero_init={_fmt(ZERO_INIT)}",
            f"model.encoder.permutation_seed={PERMUTATION_SEED}",
            f"model.encoder.use_loft={_fmt(USE_LOFT)}",
            f"model.encoder.loft_tau={LOFT_TAU}",
            f"model.n_target_dims={_fmt(N_TARGET_DIMS)}",
            f"model.use_vae={_fmt(USE_VAE)}",
            f"model.vae_sample_all_losses={_fmt(VAE_SAMPLE_ALL_LOSSES)}",
            f"model.kl_warmup_epochs={KL_WARMUP_EPOCHS}",
            f"training.lightning.kl_null_weight={KL_NULL_WEIGHT}",
            f"training.lightning.reconstruction_mode={RECONSTRUCTION_MODE}",
            f"training.lightning.tangent_entropy_weight={TANGENT_ENTROPY_WEIGHT}",
            f"training.lightning.tangent_entropy_mode={TANGENT_ENTROPY_MODE}",
            f"training.lightning.tangent_entropy_n_samples={TANGENT_ENTROPY_N_SAMPLES}",
        ]

    # Common from-scratch overrides
    overrides += [
        f"model.decode_only_recent={_fmt(DECODE_ONLY_RECENT)}",
        f"model.jac_window_stride={JAC_WINDOW_STRIDE}",
        f"model.encoder_warmup_epochs={ENCODER_WARMUP_EPOCHS}",
        f"model.dynamics_warmup_epochs={DYNAMICS_WARMUP_EPOCHS}",
        f"training.lightning.learn_r2_weight={_fmt(LEARN_R2_WEIGHT)}",
        f"training.lightning.learn_loop_closure_weight={_fmt(LEARN_LOOP_CLOSURE_WEIGHT)}",
        f"training.lightning.learn_fnn_weight={_fmt(LEARN_FNN_WEIGHT)}",
        f"training.lightning.learn_jac_cons_weight={_fmt(LEARN_JAC_CONS_WEIGHT)}",
        f"training.lightning.learn_jac_norm_weight={_fmt(LEARN_JAC_NORM_WEIGHT)}",
        f"training.lightning.log_var_init={LOG_VAR_INIT}",
        f"training.logger.save_dir={SAVE_DIR}",
    ]

# ---- Optuna pruning ----
if OPTUNA_PRUNE_EPOCH is not None:
    overrides += [
        f"training.optuna_prune_epoch={OPTUNA_PRUNE_EPOCH}",
        f"training.optuna_prune_min_completed={OPTUNA_PRUNE_MIN_COMPLETED}",
        f"training.optuna_prune_quantile={OPTUNA_PRUNE_QUANTILE}",
        f"optuna_study_name={STUDY_NAME}",
        f"optuna_storage={OPTUNA_STORAGE}",
    ]

# ---- Optuna constrained optimization ----
if OPTUNA_CONSTRAINT_METRIC is not None:
    overrides += [
        f"training.optuna_constraint_metric={OPTUNA_CONSTRAINT_METRIC}",
        f"training.optuna_constraint_threshold={OPTUNA_CONSTRAINT_THRESHOLD}",
    ]

# ---- W&B + SLURM ----
overrides += [
    f"wandb_entity={WANDB_ENTITY}",
    f"wandb_project={WANDB_PROJECT}",
    f"wandb_group={WANDB_GROUP}",
    "slurm=default",
]

print(f"Total fixed overrides: {len(overrides)}")

Total fixed overrides: 80


In [11]:
# ============================================================
# Build the coordinator Python script
# ============================================================
#
# Instead of Hydra --multirun (which batches jobs and blocks),
# we use OptunaCoordinator for true backfilling: as soon as one
# SLURM job finishes, a new trial is immediately queued.

ENTRY_POINT = (
    "python -m JacobianODE.jacobians.run_jacobians"
    if MODE == "from_scratch"
    else "python -m JacobianODE.jacobians.run_pretrained_jacobians"
)

# Build the coordinator Python code (will be written to disk and
# executed by the SLURM coordinator job).
_coordinator_py = f'''
import logging, sys
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
)

from JacobianODE.jacobians.training.optuna_coordinator import OptunaCoordinator

coordinator = OptunaCoordinator(
    fixed_overrides={overrides!r},
    search_space={SEARCH_SPACE!r},
    study_name="{STUDY_NAME}",
    storage="{OPTUNA_STORAGE}",
    entry_point="{ENTRY_POINT}",
    repo_root="{REPO_ROOT}",
    sampler_seed={SAMPLER_SEED},
    n_startup_trials={N_STARTUP_TRIALS},
    # SLURM worker params
    slurm_partition="ou_bcs_normal",
    slurm_gpus_per_node=1,
    slurm_cpus_per_task=4,
    slurm_mem_gb=16,
    slurm_timeout_min=180,
    slurm_exclude="node4000",
    poll_interval=30.0,
    # Constraint
    constraint_metric={f'"{OPTUNA_CONSTRAINT_METRIC}"' if OPTUNA_CONSTRAINT_METRIC else 'None'},
    constraint_threshold={OPTUNA_CONSTRAINT_THRESHOLD if OPTUNA_CONSTRAINT_METRIC else 'None'},
)

coordinator.run(n_trials={N_TRIALS}, n_parallel={N_PARALLEL_JOBS})
'''

# Write to disk
_coordinator_py_path = OPTUNA_SAVE_DIR / f"coordinator_{STUDY_NAME}.py"
os.makedirs(OPTUNA_SAVE_DIR, exist_ok=True)
with open(_coordinator_py_path, "w") as f:
    f.write(_coordinator_py)

print(f"Coordinator script: {_coordinator_py_path}")
print(f"\nWill run {N_TRIALS} trials, {N_PARALLEL_JOBS} parallel (true backfilling)")
print(f"Entry point: {ENTRY_POINT}")
print(f"Search space: {list(SEARCH_SPACE.keys())}")
if OPTUNA_CONSTRAINT_METRIC:
    print(f"Constraint: {OPTUNA_CONSTRAINT_METRIC} <= {OPTUNA_CONSTRAINT_THRESHOLD:.4f}")
print("to change to optuna directory:")
print(f"cd {OPTUNA_SAVE_DIR}")

Coordinator script: /orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/optuna/coordinator_optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50.py

Will run 40 trials, 10 parallel (true backfilling)
Entry point: python -m JacobianODE.jacobians.run_jacobians
Search space: ['training.lightning.loop_closure_weight', 'training.lightning.kl_dyn_weight', 'training.lightning.optimizer_kwargs.lr']
Constraint: val/loop_closure_loss <= 1.7321
to change to optuna directory:
cd /orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/optuna


## 6. Launch Sweep

The notebook submits a **coordinator SLURM job** (CPU-only, 24 h) that:
1. Uses `OptunaCoordinator` with **true backfilling** — as soon as one GPU job finishes, a new trial is immediately queued
2. Maintains `N_PARALLEL_JOBS` concurrent GPU workers at all times
3. Reads results from the shared SQLite DB (written by training callbacks)
4. Supports resuming — re-run with the same `STUDY_NAME` to continue

In [12]:
# ============================================================
# Build coordinator SLURM script
# ============================================================
COORDINATOR_TIME   = "24:00:00"   # 24 h — enough for N_TRIALS × 3 h / N_PARALLEL_JOBS
COORDINATOR_PARTITION = "ou_bcs_normal"
COORDINATOR_MEM    = "8G"
COORDINATOR_CPUS   = 2

_script_path = OPTUNA_SAVE_DIR / f"optuna_coordinator_{STUDY_NAME}.sh"
_log_path    = OPTUNA_SAVE_DIR / f"optuna_coordinator_{STUDY_NAME}_%j.log"

coordinator_script = f"""#!/bin/bash
#SBATCH --job-name=optuna_{STUDY_NAME[:30]}
#SBATCH --partition={COORDINATOR_PARTITION}
#SBATCH --time={COORDINATOR_TIME}
#SBATCH --mem={COORDINATOR_MEM}
#SBATCH --cpus-per-task={COORDINATOR_CPUS}
#SBATCH --gpus=0
#SBATCH --output={_log_path}

# ---- Environment setup ----
cd {REPO_ROOT}
source .venv/bin/activate

# ---- Run Optuna coordinator (submits GPU worker jobs via submitit) ----
python {_coordinator_py_path}
"""

print(f"Coordinator script: {_script_path}")
print(f"Log file:           {_log_path}")
print(f"\n--- Script preview ---")
print(coordinator_script)

Coordinator script: /orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/optuna/optuna_coordinator_optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50.sh
Log file:           /orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/optuna/optuna_coordinator_optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50_%j.log

--- Script preview ---
#!/bin/bash
#SBATCH --job-name=optuna_optuna_from_scratch_spline_cou
#SBATCH --partition=ou_bcs_normal
#SBATCH --time=24:00:00
#SBATCH --mem=8G
#SBATCH --cpus-per-task=2
#SBATCH --gpus=0
#SBATCH --output=/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/optuna/optuna_coordinator_optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50_%j.log

# ---- Environment setup ----
cd /orcd/home/002/eisenaj/code/JacobianODE
source .venv/bin/activate

# ---- Run Optuna coordinator (subm

In [13]:
# ============================================================
# Submit coordinator job
# ============================================================
# DRY_RUN = True  # <-- Set to False to actually submit
DRY_RUN = False

# Write script to disk
os.makedirs(Path(_script_path).parent, exist_ok=True)
with open(_script_path, "w") as f:
    f.write(coordinator_script)
os.chmod(_script_path, 0o755)

if DRY_RUN:
    print("DRY RUN — set DRY_RUN = False to submit.")
    print(f"\nTo submit manually:\n  sbatch {_script_path}")
else:
    result = subprocess.run(["sbatch", str(_script_path)], capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode != 0:
        print(f"ERROR: {result.stderr.strip()}")
    else:
        print(f"\nCoordinator submitted. Monitor with:")
        print(f"  squeue -u $USER")
        print(f"  tail -f {str(_log_path).replace('%j', '*')}")

Submitted batch job 11142881

Coordinator submitted. Monitor with:
  squeue -u $USER
  tail -f /orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/latent_jac_runs/optuna/optuna_coordinator_optuna_from_scratch_spline_coupling_loop_closure_weight_kl_dyn_weight_lr_n40_2026-03-29_01-43-50_*.log


## 7. Analyze Optuna Results

After the sweep completes, load the Optuna study to inspect trial results,
parameter importances, and optimization history.

In [ ]:
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
)

In [ ]:
# Load study (requires OPTUNA_STORAGE to have been set, or find the auto-generated db)
if OPTUNA_STORAGE is not None:
    study = optuna.load_study(study_name=STUDY_NAME, storage=OPTUNA_STORAGE)
else:
    # Without persistent storage, results are only in W&B.
    # Use the W&B-based analysis in the next section instead.
    print("No persistent storage configured — skipping Optuna study analysis.")
    print("Set OPTUNA_STORAGE = 'sqlite:///optuna_sweep.db' for study persistence.")
    study = None

In [ ]:
if study is not None:
    print(f"Study: {study.study_name}")
    print(f"Completed trials: {len(study.trials)}")
    print(f"Best trial:")
    print(f"  Value (trajectory val_loss): {study.best_value:.6f}")
    print(f"  Params: {study.best_trial.params}")
    print(f"\nTop 5 trials:")
    df = study.trials_dataframe().sort_values("value")
    display(df.head())

In [ ]:
if study is not None:
    # Optimization history: shows convergence over trials
    plot_optimization_history(study)

In [ ]:
if study is not None:
    # Parameter importances: which params matter most
    plot_param_importances(study)

In [ ]:
if study is not None:
    # Slice plot: objective vs each param
    plot_slice(study)

In [ ]:
if study is not None:
    # Parallel coordinate plot: see param interactions
    plot_parallel_coordinate(study)

## 8. Post-Hoc Physics Selection

Apply the standard physics-based criteria (C1–C3) to the Optuna-optimized runs.
This reuses the existing selection pipeline from `JacobianODE.jacobians.tuning`.

In [ ]:
from JacobianODE.jacobians.tuning import select_best_from_sweep

In [ ]:
best_run_id, sweep_result, discovered = select_best_from_sweep(
    wandb_entity=WANDB_ENTITY,
    wandb_project=WANDB_PROJECT,
    save_dir="./sweep_results",
    wandb_group=WANDB_GROUP,
    verbose=True,
)

if best_run_id:
    print(f"\nBest run: {best_run_id}")
else:
    print("\nNo runs passed all physics criteria.")